# 3.10 状态空间模型与混合架构 (SSM & Hybrid)

> 🕐 预估学习时间：40分钟

Mamba / Mamba-2、RWKV、RetNet 等线性复杂度序列模型，以及 Jamba、Zamba、Jamba-1.5 等 Attention+SSM 混合架构，是长上下文与高吞吐场景的重要选项。

本节涵盖：
- 离散状态空间与选择性扫描
- Mamba 块结构（简化实现）
- Attention–SSM 混合堆叠
- 与 Transformer 的效率/质量权衡


## 1. 从 S4 到 Selective SSM

经典 SSM：`h_t = A h_{t-1} + B x_t`, `y_t = C h_t + D x_t`

**选择性（Mamba）关键改动**：让 `B/C/Δ` 依赖输入，使模型能对当前 token **选择记住或忘记**，大幅提升语言建模质量。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

torch.manual_seed(42)


class MambaBlock(nn.Module):
    '''Educational selective SSM block (scan implemented in Python for clarity).'''
    def __init__(self, d_model=64, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = expand * d_model
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, d_conv, padding=d_conv - 1, groups=self.d_inner)
        self.x_proj = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)
        self.dt_proj = nn.Linear(1, self.d_inner)
        self.A_log = nn.Parameter(torch.log(torch.rand(self.d_inner, d_state) + 0.1))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def ssm_scan(self, u, delta, B, C):
        # u, delta: (B, T, D); B,C: (B, T, N)
        batch, seq, d = u.shape
        A = -torch.exp(self.A_log)  # (D, N)
        h = torch.zeros(batch, d, self.d_state, device=u.device)
        ys = []
        for t in range(seq):
            dt = delta[:, t].unsqueeze(-1)          # (B, D, 1)
            A_bar = torch.exp(A.unsqueeze(0) * dt)  # (B, D, N)
            B_t = B[:, t].unsqueeze(1)              # (B, 1, N)
            C_t = C[:, t].unsqueeze(1)
            h = h * A_bar + u[:, t].unsqueeze(-1) * B_t * dt
            y = (h * C_t).sum(-1) + self.D * u[:, t]
            ys.append(y)
        return torch.stack(ys, dim=1)

    def forward(self, x):
        batch, seq, _ = x.shape
        xz = self.in_proj(x)
        x_branch, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_branch.transpose(1, 2))[:, :, :seq].transpose(1, 2)
        x_conv = F.silu(x_conv)
        params = self.x_proj(x_conv)
        B = params[..., :self.d_state]
        C = params[..., self.d_state:2 * self.d_state]
        dt = F.softplus(self.dt_proj(params[..., -1:].contiguous()))
        y = self.ssm_scan(x_conv, dt, B, C)
        y = y * F.silu(z)
        return self.out_proj(y)


x = torch.randn(2, 128, 64)
mamba = MambaBlock()
y = mamba(x)
print('=== Mamba Block ===')
print(f'Input {tuple(x.shape)} -> Output {tuple(y.shape)}')
print(f'Params: {sum(p.numel() for p in mamba.parameters()):,}')
print(f'Key: Selective SSM keeps O(n) compute/memory while making B/C/Δ input-dependent.')


## 2. 复杂度对比：Attention vs SSM

| 方法 | 训练计算 | 推理显存 (KV/state) | 质检长程依赖 |
|------|---------|---------------------|-------------|
| MHA | O(n²d) | O(n d) KV | 强 |
| GQA/MQA | O(n²d) | 更低 KV | 强 |
| Mamba | O(n d r) | O(d r) 固定状态 | 中-强 |
| Hybrid | 介于两者 | 介于两者 | 通常最好 |

结论：纯 SSM 吞吐优势大；需要精确召回针点信息时，混合架构更稳。


In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, d=64, h=4):
        super().__init__()
        self.ln = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, h, batch_first=True)
        self.mlp = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))

    def forward(self, x):
        h = self.ln(x)
        a, _ = self.attn(h, h, h, need_weights=False)
        x = x + a
        return x + self.mlp(x)


def bench(module, seq_lens, d=64, batch=2, warmup=1):
    results = []
    for n in seq_lens:
        x = torch.randn(batch, n, d)
        for _ in range(warmup):
            module(x)
        t0 = time.perf_counter()
        for _ in range(3):
            module(x)
        dt = (time.perf_counter() - t0) / 3
        results.append((n, dt))
    return results


attn = AttentionBlock()
ssm = MambaBlock()
seqs = [64, 128, 256, 512]
print('=== Runtime scaling (CPU educational) ===')
print(f'{"N":>6} {"Attn(s)":>10} {"Mamba(s)":>10} {"Attn/Mamba":>12}')
for (n, ta), (_, tm) in zip(bench(attn, seqs), bench(ssm, seqs)):
    print(f'{n:>6} {ta:>10.4f} {tm:>10.4f} {ta/max(tm,1e-6):>12.2f}x')
print(f'\nKey: Attention grows closer to quadratic; SSM scan is linear in sequence length.')


## 3. Hybrid：交错堆叠 Attention 与 SSM

Jamba 风格：多数层用 Mamba，每隔若干层插入注意力层，兼顾全局精确匹配与线性吞吐。


In [ ]:
class HybridModel(nn.Module):
    def __init__(self, d=64, n_layers=6, attn_every=3, vocab=100):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.layers = nn.ModuleList()
        self.types = []
        for i in range(n_layers):
            if (i + 1) % attn_every == 0:
                self.layers.append(AttentionBlock(d))
                self.types.append('attn')
            else:
                self.layers.append(MambaBlock(d))
                self.types.append('mamba')
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab)

    def forward(self, ids):
        x = self.embed(ids)
        for layer in self.layers:
            x = layer(x)
        return self.head(self.ln(x))


hybrid = HybridModel()
ids = torch.randint(0, 100, (2, 64))
logits = hybrid(ids)
print('=== Hybrid Attention+SSM ===')
print(f'Layer schedule: {hybrid.types}')
print(f'Logits: {tuple(logits.shape)}')
print(f'Total params: {sum(p.numel() for p in hybrid.parameters()):,}')
print(f'\nKey: Put scarce Attention layers where precise token-token routing matters;')
print(f'use SSM layers for cheap long-range state propagation.')


## 课后思考题

1. 选择性机制为什么比固定 A/B/C 的经典 SSM 更适合语言建模？
2. 在 1M 上下文服务中，何时优先选 Hybrid 而非纯 Transformer + 稀疏注意力？
3. 混合架构如何与 GQA、量化、推测解码组合？
4. 教育实现里的 Python scan 与硬件高效 parallel scan 差在哪里？

---
> 本节涵盖了3.10 状态空间模型与混合架构的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
